# Databricks notebook source

## 1. Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from scipy.stats import gaussian_kde
import plotly.express as px # Used for color palette only
import requests
from scipy import stats

# Notebook configuration
%matplotlib inline
%config InlineBackend.figure_format = 'retina'
sns.set_theme(style="whitegrid", palette="pastel")
import warnings
warnings.filterwarnings('ignore')


## 2. Data Import and Preparation

In [ ]:
# Target table containing elasticity data
table_target = 'hive_metastore.analytics.discount_elasticity_trusted'

# Source table for historical margin data
table_source = 'analytics.discount_raw_data'

# SQL Query to fetch all data from target table,
# adding 'previous_theoretical_unit_margin' from source table.
# Using LEFT JOIN to ensure all records from 'table_target' are kept.
# Join performed on a robust set of keys to ensure accuracy.
query = f"""
SELECT
    t.*, -- Select all columns from target table
    s.previous_theoretical_unit_margin -- Add desired column from source table
    --s.total_unit_cogs
FROM
    {table_target} AS t
LEFT JOIN
    {table_source} AS s
ON
    -- Primary keys: Transaction and Product
    t.ORDER_ID = s.ORDER_ID
    AND t.PRODUCT_ID = s.PRODUCT_ID
    
    -- Date keys (available in both tables)
    AND t.month = s.month
    AND t.week_of_year = s.week_of_year
    AND t.day_of_week = s.day_of_week
    
    -- Business keys and context to ensure uniqueness
    AND t.CAMPAIGN_ID = s.CAMPAIGN_ID
    AND t.STATE = s.STATE
    AND t.Sub_Channel = s.Sub_Channel
    AND t.sales_team_channel_name = s.sales_team_channel_name
    AND t.Product_Category = s.Product_Category
    AND t.BUSINESS_UNIT = s.BUSINESS_UNIT
    AND t.SUB_BUSINESS_UNIT = s.SUB_BUSINESS_UNIT
    AND t.BRAND = s.BRAND
"""

# Import query result to a pandas dataframe
print("Executing query with JOIN to fetch 'previous_theoretical_unit_margin'...")
df_data = spark.sql(query).toPandas()
print("Query finished. Data loaded into Pandas.")


# Format adjustments
df_data['Product_Category_reduced'] = df_data['Product_Category_reduced'].astype(str)
df_data['theoretical_unit_margin'] = df_data['theoretical_unit_margin'].astype(float)

# Verification of the new column
if 'previous_theoretical_unit_margin' in df_data.columns:
    print("\nColumn 'previous_theoretical_unit_margin' added successfully.")
    
    # Convert new column to float, handling errors
    df_data['previous_theoretical_unit_margin'] = pd.to_numeric(df_data['previous_theoretical_unit_margin'], errors='coerce')
    
    null_count = df_data['previous_theoretical_unit_margin'].isnull().sum()
    total_count = len(df_data)
    print(f"Matched {total_count - null_count} values out of {total_count} total records.")
    
    if null_count > 0:
        print(f"WARNING: {null_count} records did not find a match in 'raw' table and remain Null.")
else:
    print("\nERROR: Column 'previous_theoretical_unit_margin' not found after join.")


# Display final DataFrame info
df_data.info()

In [ ]:
display(df_data)


## 3. Discount Calculations

In [ ]:
def drop_proc_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Remove all columns ending with '_proc'.
    """
    proc_cols = [col for col in df.columns if col.endswith('_proc')]
    return df.drop(columns=proc_cols)

df_clean = drop_proc_columns(df_data)

df_clean.info()

In [ ]:
# 2. Interactive Filter Configuration
def create_filter_widgets(df: pd.DataFrame):
    ALL = "All"
    dbutils.widgets.multiselect("filter_state", ALL, [ALL] + sorted(df["STATE"].unique().tolist()))
    dbutils.widgets.multiselect("filter_region", ALL, [ALL] + sorted(df["Region"].unique().tolist()))
    dbutils.widgets.multiselect("filter_channel", ALL, [ALL] + sorted(df["SALES_CHANNEL_TEAM"].unique().tolist()))
    dbutils.widgets.multiselect("filter_subchannel", ALL, [ALL] + sorted(df["Sub_Channel"].unique().tolist()))
    dbutils.widgets.multiselect("filter_sales_team", ALL, [ALL] + sorted(df["sales_team_channel_name"].unique().tolist()))
    dbutils.widgets.multiselect("filter_cat", ALL, [ALL] + sorted(df["Product_Category"].unique().tolist()))
    dbutils.widgets.multiselect("filter_pid", ALL, [ALL] + sorted(df["PRODUCT_ID"].unique().tolist()))
    dbutils.widgets.multiselect("filter_brand", ALL, [ALL] + sorted(df["BRAND"].unique().tolist()))
    dbutils.widgets.multiselect("filter_bu", ALL, [ALL] + sorted(df["BUSINESS_UNIT"].unique().tolist()))
    dbutils.widgets.multiselect("filter_sub_bu", ALL, [ALL] + sorted(df["SUB_BUSINESS_UNIT"].unique().tolist()))
    dbutils.widgets.multiselect("filter_month", ALL, [ALL] + sorted(df["month"].astype(str).unique().tolist()))

    # 2.1) Numeric widgets for lift_min and margin_min
    dbutils.widgets.text("lift_min", "0.05", "Minimum Volume Uplift Threshold (e.g., 0.05)")
    dbutils.widgets.text("margin_min", "0.05", "Minimum Margin (e.g., 0.05)")

# 3. Reading and parsing selectors
def parse_widget_selection(widget_name: str, cast_type=None):
    raw = dbutils.widgets.get(widget_name)
    selections = [s for s in raw.split(",") if s]
    if not selections or "All" in selections:
        return None
    return [cast_type(s) if cast_type else s for s in selections]

def get_filters():
    return {
        "STATE": parse_widget_selection("filter_state"),
        "Region": parse_widget_selection("filter_region"),
        "SALES_CHANNEL_TEAM": parse_widget_selection("filter_channel"),
        "Sub_Channel": parse_widget_selection("filter_subchannel"),
        "sales_team_channel_name": parse_widget_selection("filter_sales_team"),
        "Product_Category": parse_widget_selection("filter_cat"),
        "PRODUCT_ID": parse_widget_selection("filter_pid"),
        "BRAND": parse_widget_selection("filter_brand"),
        "BUSINESS_UNIT": parse_widget_selection("filter_bu"),
        "SUB_BUSINESS_UNIT": parse_widget_selection("filter_sub_bu"),
        "month": parse_widget_selection("filter_month", cast_type=int),
    }

# 4. Applying filters to DataFrame
def apply_filters(df: pd.DataFrame, filters: dict) -> pd.DataFrame:
    mask = pd.Series(True, index=df.index)
    for col, sel in filters.items():
        if sel is not None:
            mask &= df[col].isin(sel)
    return df.loc[mask]

In [ ]:
def generate_histogram(
    df: pd.DataFrame, 
    numeric_col: str, 
    categorical_col: str = None, 
    width: int = None, 
    height: int = None,
    nbins: int = 50,
    show_kde: bool = False,
    show_bars: bool = True):
    """
    Generates a density plot with options to show/hide bars and KDE curves.

    Args:
        df (pd.DataFrame): DataFrame containing data.
        numeric_col (str): Name of numeric column.
        categorical_col (str, optional): Column name for grouping/coloring. Defaults to None.
        width (int, optional): Figure width in pixels. Defaults to None.
        height (int, optional): Figure height in pixels. Defaults to None.
        nbins (int): Number of histogram bins. Default is 50.
        show_kde (bool, optional): If True, shows Kernel Density Estimation curve. Defaults to False.
        show_bars (bool, optional): If True, shows histogram bars. Defaults to True.
    """
    # Validations
    if not show_bars and not show_kde:
        print("Error: Nothing to display. Set 'show_bars' or 'show_kde' to True.")
        return
    if numeric_col not in df.columns:
        print(f"Error: Numeric column '{numeric_col}' not found.")
        return
    if categorical_col and categorical_col not in df.columns:
        print(f"Error: Categorical column '{categorical_col}' not found.")
        return
    if nbins is None: nbins = 50

    fig = go.Figure()
    colors = px.colors.qualitative.Plotly
    
    categories = sorted(df[categorical_col].unique()) if categorical_col else [None]

    for i, category in enumerate(categories):
        
        if category is not None:
            group_data = df[df[categorical_col] == category][numeric_col]
            group_name = str(category)
            current_color = colors[i % len(colors)]
        else:
            group_data = df[numeric_col]
            group_name = "Data"
            current_color = colors[0]

        if len(group_data) < 2:
            continue

        # Logic to show histogram bars
        if show_bars:
            # --- MANUAL DENSITY CALCULATION ---
            counts, bin_edges = np.histogram(group_data, bins=nbins, density=False)
            bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
            bin_width = bin_edges[1] - bin_edges[0]
            densities = counts / (np.sum(counts) * bin_width)

            # Add bars using go.Bar
            fig.add_trace(go.Bar(
                x=bin_centers,
                y=densities,
                name=group_name,
                marker_color=current_color))

        # Logic to show KDE curve
        if show_kde:
            kde = gaussian_kde(group_data)
            x_range = np.linspace(group_data.min(), group_data.max(), 200)
            fig.add_trace(go.Scatter(
                x=x_range,
                y=kde(x_range),
                mode='lines',
                name=f'KDE {group_name}',
                line=dict(color=current_color, width=3)))

    # --- Final Layout Config ---
    # Dynamic title based on display
    if show_bars:
        title = f'Distribution of {numeric_col}'
    else:
        title = f'Density Curves (KDE) of {numeric_col}'
        
    if categorical_col:
        title += f' by {categorical_col}'
    
    fig.update_layout(
        title_text=title,
        xaxis_title_text='Value',
        yaxis_title_text='Density',
        barmode='overlay',
        width=width,
        height=height,
        template='plotly_white',
        font_family="Arial",
        legend_title_text='Legend')
    
    # Apply opacity only if bars are visible
    if show_bars:
        fig.update_traces(opacity=0.6, selector=dict(type='bar'))
        
    fig.show()



def generate_geo_map(
    df: pd.DataFrame, 
    state_col: str, 
    numeric_col: str,
    color_scale: str = 'RdYlBu_r',
    height: int = 600,
    width: int = None):
    """
    Generates a choropleth map (focused on Brazil states) from a DataFrame.

    Aggregates data by state, calculates mean of numeric column,
    and plots result.

    Args:
        df (pd.DataFrame): Data. Must include State acronym column and numeric column.
        state_col (str): Column name containing state acronyms (e.g., 'STATE').
        numeric_col (str): Numeric column name to plot.
        color_scale (str, optional): Color scale. Default 'RdYlBu_r'.
        height (int, optional): Height in pixels. Default 600.
        width (int, optional): Width in pixels. Default None.
    """
    # -- 1. Data Prep --
    if state_col not in df.columns or numeric_col not in df.columns:
        print(f"Error: Columns '{state_col}' and/or '{numeric_col}' not found.")
        return

    df_map = df[[state_col, numeric_col]] \
        .groupby(state_col) \
        .mean() \
        .reset_index()

    # -- 2. Get GeoJSON --
    try:
        url = "https://raw.githubusercontent.com/codeforamerica/click_that_hood/master/public/data/brazil-states.geojson"
        response = requests.get(url)
        response.raise_for_status()
        geojson_br = response.json()
    except requests.exceptions.RequestException as e:
        print(f"Error downloading GeoJSON: {e}")
        return

    # -- 3. Generate Map --
    chart_title = f'{numeric_col.replace("_", " ").title()} Mean by State'
    
    fig = px.choropleth(
        data_frame=df_map,
        geojson=geojson_br,
        locations=state_col,
        featureidkey='properties.sigla',
        color=numeric_col,
        color_continuous_scale=color_scale,
        scope='south america',
        title=chart_title,
        labels={numeric_col: 'Mean'})

    # Center map on region
    fig.update_geos(
        center={"lat": -14.2350, "lon": -51.9253},
        lataxis_range=[-35, 6],
        lonaxis_range=[-75, -30],
        visible=False)

    # Layout adjustment
    fig.update_layout(
        title_font_size=20,
        title_x=0.5,
        margin={"r":0, "t":50, "l":0, "b":0},
        height=height,
        width=width)

    fig.show()

In [ ]:
def plot_boxplot_with_stats(df, column='ORDER_DISCOUNT'):
    """
    Generates an interactive HORIZONTAL boxplot with Plotly for the given column
    and annotates key statistics (min, Q1, median, mean, Q3, max),
    ensuring labels don't overlap but stay aligned to values.
    
    Params:
    - df: pandas.DataFrame containing interest column.
    - column: numeric column name containing discount values.
    """
    # 1) Extract values and round to 2 decimals
    vals = df[column].round(2).values

    # 2) Calculate key stats
    stats_calc = [
        ('max',    np.max(vals)),
        ('q3',     np.percentile(vals, 75)),
        ('median', np.percentile(vals, 50)),
        ('mean',   np.mean(vals)),
        ('q1',     np.percentile(vals, 25)),
        ('min',    np.min(vals))]

    # 3) Define vertical positions spaced on paper for annotations
    n = len(stats_calc)
    y_positions = np.linspace(0.7, 0.98, n)

    # 4) Create annotation list
    annotations = []
    for (label, value), y in zip(stats_calc, y_positions):
        annotations.append(dict(
            x=value,        # align on X axis to exact stat value
            xref='x',
            y=y,            # distinct vertical position
            yref='paper',
            text=f"{label}: {value:.2f}",
            showarrow=False,
            xanchor='center',
            yanchor='bottom',
            font=dict(size=12)))

    # 5) Create horizontal boxplot without internal points
    fig = go.Figure(go.Box(
        x=vals,
        name=column,
        boxpoints=False,
        orientation='h'))

    # 6) Adjust layout
    fig.update_layout(
        title=f"Historical Segment Discounts",
        xaxis_title=column,
        yaxis=dict(showticklabels=False),
        annotations=annotations,
        height=400,
        width=1800)

    # 7) Show
    fig.show()

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def simulate_elasticity(base_price, cogs, elasticity, distributor_discount, max_discount_perc=50, steps=100):
    """
    Simulates discount impact on margin, uplift, and total profit (index).
    """
    net_price_base = base_price * (1 - 0)
    gross_profit_base = net_price_base - cogs
    base_volume_index = 1.0
    constant_price = base_price * (1 - distributor_discount)
    total_profit_base = gross_profit_base * base_volume_index
    
    discount_range = np.linspace(0, max_discount_perc / 100.0, steps)
    results = []
    for d in discount_range:
        net_price_new = base_price * (1 - d)
        gross_profit_new = net_price_new - cogs
        
        if constant_price == 0:
            new_gross_margin = 0
        else:
            new_gross_margin = gross_profit_new / constant_price
            
        perc_price_delta = (net_price_new - net_price_base) / net_price_base
        perc_volume_uplift = elasticity * perc_price_delta
        
        volume_new_index = (1 - d) ** elasticity
        total_profit_new = gross_profit_new * volume_new_index
        total_profit_index = (total_profit_new / total_profit_base) * 100
        
        results.append({
            "Discount_Perc": d * 100,
            "Volume_Uplift_Perc": perc_volume_uplift * 100,
            "Gross_Margin_Perc": new_gross_margin * 100,
            "Total_Profit_Index": total_profit_index
        })
    return pd.DataFrame(results)


import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots


def plot_simulation_plotly(df, d_calc=None, d_star=None, d_min=None, d_max=None, lift_min=None, margin_min=None, 
                            graph_width=1000, graph_height=600):
    """
    Generates interactive plot with all series and vertical lines.
    Adjusts Y-axis padding.
    """

    if d_calc is not None:
        optimal_point = df.loc[df['Total_Profit_Index'].idxmax()]
        optimal_profit = optimal_point['Total_Profit_Index']
    
    fig = make_subplots(specs=[[{"secondary_y": True}]])
    
    # --- 1. EXPANDED RANGE CALCULATION ---
    
    Y_PADDING = 20 
    Y_MIN_AXIS = df['Volume_Uplift_Perc'].min() - Y_PADDING
    Y_MAX_AXIS = df['Volume_Uplift_Perc'].max() + Y_PADDING 

    Y_TRACE_MIN = Y_MIN_AXIS - 10 
    Y_TRACE_MAX = Y_MAX_AXIS + 10 

    # --- 2. MAIN CURVES ---
    
    fig.add_trace(go.Scatter(x=df['Discount_Perc'], y=df['Volume_Uplift_Perc'], mode='lines', name='Volume Uplift (%)', line=dict(color='blue', dash='dash')), secondary_y=False)
    fig.add_trace(go.Scatter(x=df['Discount_Perc'], y=df['Gross_Margin_Perc'], mode='lines', name='Gross Margin (%)', line=dict(color='orange', dash='dash')), secondary_y=False)
    fig.add_trace(go.Scatter(x=df['Discount_Perc'], y=df['Total_Profit_Index'], mode='lines', name='Total Profit (Index)', line=dict(color='green', width=3)), secondary_y=True)

    if d_calc is not None and 'optimal_profit' in locals():
        fig.add_trace(go.Scatter(x=[d_calc], y=[optimal_profit], mode='markers', name='Max Profit (Calc)', marker=dict(symbol='circle', size=10, color='lightgreen', line=dict(width=1, color='green')), showlegend=True), secondary_y=True)
    
    # Base Profit Line (Index 100)
    fig.add_trace(go.Scatter(x=df['Discount_Perc'], y=[100]*len(df), mode='lines', line=dict(color='grey', dash='dot'), name='Base Profit (Index 100)', showlegend=True), secondary_y=True)
    
    # --- CRITICAL ADJUSTMENT: Force expanded visible range for primary Y axis ---
    fig.update_yaxes(range=[Y_MIN_AXIS, Y_MAX_AXIS], secondary_y=False) 

    # --- 3. VERTICAL LINES (Traces) and Annotations ---
    
    # 1. Max Gross Profit Discount (Calculated - Red)
    if d_calc is not None:
        name_calc = f'Max GP Discount ({d_calc:.1f}%)'
        fig.add_trace(go.Scatter(x=[d_calc, d_calc], y=[Y_TRACE_MIN, Y_TRACE_MAX], mode='lines', 
                                 line=dict(color='red', width=1.5, dash='dash'), 
                                 name=name_calc, showlegend=True, hoverinfo='skip'),
                      secondary_y=False) 

        fig.add_annotation(x=d_calc, y=Y_MAX_AXIS, yref='y1', text=f'{d_calc:.1f}%', 
                           showarrow=False, font=dict(color='red', size=9), 
                           xanchor='left', yanchor='top', yshift=-5)


    # 2. Theoretical Optimal Discount (d_star - Magenta)
    if d_star is not None:
        name_star = f'Theoretical Optimal Disc. (δ* = {d_star:.1f}%)'
        fig.add_trace(go.Scatter(x=[d_star, d_star], y=[Y_TRACE_MIN, Y_TRACE_MAX], mode='lines', 
                                 line=dict(color='magenta', width=1.5, dash='dot'), 
                                 name=name_star, showlegend=True, hoverinfo='skip'),
                      secondary_y=False)

        fig.add_annotation(x=d_star, y=Y_MAX_AXIS, yref='y1', text=f'{d_star:.1f}%', 
                           showarrow=False, font=dict(color='magenta', size=9), 
                           xanchor='left', yanchor='top', yshift=-25)


    # 3. Minimum Discount (d_min - Purple)
    if d_min is not None and lift_min is not None:
        name_min = f'Min Disc. (Target Lift = {lift_min * 100:.1f}%)'
        fig.add_trace(go.Scatter(x=[d_min, d_min], y=[Y_TRACE_MIN, Y_TRACE_MAX], mode='lines', 
                                 line=dict(color='purple', width=1.5, dash='dot'), 
                                 name=name_min, showlegend=True, hoverinfo='skip'),
                      secondary_y=False)

        fig.add_annotation(x=d_min, y=Y_MAX_AXIS, yref='y1', text=f'{d_min:.1f}%', 
                           showarrow=False, font=dict(color='purple', size=9), 
                           xanchor='left', yanchor='top', yshift=-45)

    # 4. Maximum Discount (d_max - Brown)
    if d_max is not None and margin_min is not None:
        name_max = f'Max Disc. (Min Margin = {margin_min * 100:.1f}%)'
        fig.add_trace(go.Scatter(x=[d_max, d_max], y=[Y_TRACE_MIN, Y_TRACE_MAX], mode='lines', 
                                 line=dict(color='brown', width=1.5, dash='dot'), 
                                 name=name_max, showlegend=True, hoverinfo='skip'),
                      secondary_y=False)

        fig.add_annotation(x=d_max, y=Y_MAX_AXIS, yref='y1', text=f'{d_max:.1f}%', 
                           showarrow=False, font=dict(color='brown', size=9), 
                           xanchor='left', yanchor='top', yshift=-65)
        
    # --- 4. LAYOUT AND LABELS ---
    fig.update_layout(
        width=graph_width, 
        height=graph_height,
        title_text='Discount Simulation vs. Volume & Total Profit',
        xaxis_title_text='Applied Discount (%)',
        hovermode="x unified",
        legend=dict(x=0.5, y=-0.2, xanchor='center', yanchor='top', orientation='h', itemclick='toggle', itemdoubleclick='toggleothers'), 
        annotations=fig.layout.annotations)

    fig.update_yaxes(title_text='Uplift & Gross Margin (%)', secondary_y=False, title_font=dict(color='blue'), tickfont=dict(color='blue'))
    fig.update_yaxes(title_text='Total Profit (Index Base 100)', secondary_y=True, title_font=dict(color='green'), tickfont=dict(color='green'))

    return fig

def plot_simulation(df, d_calc=None, d_star=None, d_min=None, d_max=None, lift_min=None, margin_min=None, brands='GENERIC_BRAND'):
    """
    Plots simulation results using Matplotlib. Returns d_calc and figure object.
    """

    # Disables interactive mode only within this block
    with plt.ioff():

        # --- Initial Config ---
        fig, ax1 = plt.subplots(figsize=(14, 7))

        if d_calc is not None:
            optimal_point = df.loc[df['Total_Profit_Index'].idxmax()]
            optimal_profit = optimal_point['Total_Profit_Index']
            
            print(f"--- Max Gross Profit (GP) Point Found ---")
            print(f"Max GP Discount (Calculated): {d_calc:.2f}%")
            
            if lift_min is not None:
                print(f"Target Min Lift (Memory): {lift_min * 100:.1f}%")
            if margin_min is not None:
                print(f"Target Min Margin (Memory): {margin_min * 100:.1f}%")
                
            print("-------------------------------------------------------")
            
            ax2 = ax1.twinx()
            ax2.plot(d_calc, optimal_profit, 'go', markersize=10, label='Max Profit (Calc)')
        else:
            ax2 = ax1.twinx()
        
        # Axes and Base Lines
        ax1.set_xlabel('Applied Discount (%)')
        ax1.set_ylabel('Uplift & Gross Margin (%)', color='tab:blue')
        ax1.plot(df['Discount_Perc'], df['Volume_Uplift_Perc'], color='blue', linestyle='--', label='Volume Uplift (%)')
        ax1.plot(df['Discount_Perc'], df['Gross_Margin_Perc'], color='orange', linestyle='--', label='Gross Margin (%)')
        ax1.tick_params(axis='y', labelcolor='tab:blue')
        ax1.grid(axis='y', linestyle=':', alpha=0.7)

        ax2.set_ylabel('Total Profit (Index Base 100)', color='tab:green', weight='bold')
        ax2.plot(df['Discount_Perc'], df['Total_Profit_Index'], color='green', linewidth=3, label='Total Profit (Index)')
        ax2.tick_params(axis='y', labelcolor='tab:green')
        ax2.axhline(y=100, color='grey', linestyle=':', label='Base Profit (Index 100)')
        
        
        # --- Vertical Lines and Annotations ---
        
        y_max_ax1 = ax1.get_ylim()[1]
        x_padding = (ax1.get_xlim()[1] - ax1.get_xlim()[0]) * 0.01
        
        
        # 1. Max GP Discount
        if d_calc is not None:
            label_calc = f'Max GP Discount ({d_calc:.1f}%)'
            ax1.axvline(x=d_calc, color='red', linestyle='--', linewidth=1.5, label=label_calc)
            y_offset_calc = y_max_ax1 * 0.1 
            ax1.text(
                d_calc + x_padding,
                y_max_ax1 - y_offset_calc,
                f'{d_calc:.1f}%',
                ha='left',
                va='top',
                color='red',
                fontweight='bold',
                fontsize=9)
        
        # 2. Theoretical Optimal Discount
        if d_star is not None:
            label_star = f'Theoretical Optimal Disc. (δ* = {d_star:.1f}%)'
            ax1.axvline(x=d_star, color='magenta', linestyle=':', linewidth=1.5, label=label_star)
            y_offset_star = y_max_ax1 * 0.2
            ax1.text(
                d_star + x_padding,
                y_max_ax1 - y_offset_star,
                f'{d_star:.1f}%',
                ha='left',
                va='top',
                color='magenta',
                fontweight='bold',
                fontsize=9)
                     
        # 3. Min Discount
        if d_min is not None:
            label_min = f'Min Disc. (Target Lift = {lift_min * 100:.1f}%)'
            ax1.axvline(x=d_min, color='purple', linestyle=':', linewidth=1.5, label=label_min)
            
            y_offset_min = y_max_ax1 * 0.3
            ax1.text(
                d_min + x_padding,
                y_max_ax1 - y_offset_min,
                f'{d_min:.1f}%', 
                ha='left',
                va='top', 
                color='purple',
                fontweight='bold',
                fontsize=9)
                     
        # 4. Max Discount
        if d_max is not None:
            label_max = f'Max Disc. (Min Margin = {margin_min * 100:.1f}%)'
            ax1.axvline(x=d_max, color='brown', linestyle=':', linewidth=1.5, label=label_max)
            
            y_offset_max = y_max_ax1 * 0.4
            ax1.text(
                d_max + x_padding,
                y_max_ax1 - y_offset_max,
                f'{d_max:.1f}%', 
                ha='left',
                va='top', 
                color='brown',
                fontweight='bold',
                fontsize=9)
                
        # Finalization and Legend
        fig.tight_layout() 
        plt.subplots_adjust(right=0.75, top=0.92) 
        
        handles1, labels1 = ax1.get_legend_handles_labels()
        handles2, labels2 = ax2.get_legend_handles_labels()
        all_handles = handles1 + handles2
        all_labels = labels1 + labels2
        fig.legend(all_handles, all_labels, loc='center left', bbox_to_anchor=(0.8, 0.25))
        
        plt.title(f'Discount Simulation vs. Volume & Profit - {brands}')
    
    return d_calc, fig

In [ ]:
# =============================================================================
# FUNCTION DEFINITIONS
# =============================================================================

def aggregate_metrics(
    df: pd.DataFrame,
    volume_col: str = "Quantity",
    elasticity_col: str = "final_elasticity",
    margin_col: str = "theoretical_unit_margin",
    zscore_threshold: float = 3.0) -> dict:
    """
    Aggregates volume, elasticity, and margin metrics from a DataFrame.
    Calculates weighted averages using volume as weight.
    Applies Z-score outlier filtering.

    Returns:
    - dict: 'total_volume', 'avg_elasticity', 'avg_margin'.
    """
    df_clean = df.copy()

    # Calculate Z-score for main columns
    zs_volume = stats.zscore(df_clean[volume_col])
    zs_elast = stats.zscore(df_clean[elasticity_col])
    zs_margin = stats.zscore(df_clean[margin_col])

    mask = (
        (abs(zs_volume) <= zscore_threshold) &
        (abs(zs_elast) <= zscore_threshold) &
        (abs(zs_margin) <= zscore_threshold))
    df_filtered = df_clean[mask]

    total_volume = df_filtered[volume_col].sum()

    if total_volume == 0:
        return {"total_volume": 0, "avg_elasticity": 0.0, "avg_margin": 0.0}

    weighted_elasticity = (
        df_filtered[elasticity_col] * df_filtered[volume_col]).sum() / total_volume

    weighted_margin = (
        df_filtered[margin_col] * df_filtered[volume_col]).sum() / total_volume

    return {
        "total_volume": total_volume,
        "avg_elasticity": weighted_elasticity,
        "avg_margin": weighted_margin
    }


def calculate_discount_ranges(
    avg_elasticity: float,
    avg_margin: float,
    lift_min: float = 0.05,
    margin_min: float = 0.05) -> dict:
    """
    Calculates discount ranges (ideal, minimum, maximum) based on economic theory.

    Logic:
    1) Ideal Discount (delta_star):
       - If |E| > 1 (elastic): Use Lerner rule derivative to maximize profit.
       - If |E| <= 1 (inelastic): Ideal is zero discount to preserve margin.

    2) Minimum Discount (delta_min):
       Discount needed to reach volume uplift target.
       (1 + lift_min) = (1 + delta)^E

    3) Maximum Discount (delta_max):
       Limit keeping unit margin above margin_min.
       Solves for delta in margin equation.

    Returns:
       dict with delta_star, delta_min, delta_max.
    """

    E = avg_elasticity
    M = avg_margin

    # 1. Ideal Discount (delta_star)
    abs_E = abs(E)

    if abs_E > 1:
        # ELASTIC REGIME: Discounting can increase total profit.
        # delta* = - (|E| / (|E| + 1)) * M
        delta_star = - (abs_E / (abs_E + 1.0)) * M
    else:
        # INELASTIC REGIME: Volume reacts poorly. No discount recommended.
        delta_star = 0.0

    # 2. Minimum Discount for lift_min (delta_min)
    if E != 0:
        delta_min = (1 + lift_min) ** (1 / E) - 1
    else:
        delta_min = 0.0

    # 3. Maximum Discount for margin_min (delta_max)
    if (1 - margin_min) != 0:
        delta_max_raw = (M - margin_min) / (1 - margin_min)
    else:
        delta_max_raw = 0.0

    delta_max = -max(delta_max_raw, 0.0)

    return {
        "delta_star": delta_star,   
        "delta_min": delta_min,     
        "delta_max": delta_max      
    }


def predict_lift(discount: float, elasticity: float) -> float:
    """
    Predicts volume uplift percentage given discount and elasticity.
    Uplift = (1 + delta)^E - 1
    """
    return (1 + discount) ** elasticity - 1


def display_discounts(discounts: dict):
    """
    Formats and displays discounts (ideal, min, max).
    """
    df = pd.DataFrame([{
        "Ideal Discount": discounts["delta_star"],
        "Min Discount": discounts["delta_min"],
        "Max Discount": discounts["delta_max"]
    }])
    df = df.applymap(lambda x: f"{x:.2%}")
    display(df)


def compute_gp_variation(uplift: float, new_margin: float) -> float:
    """
    Calculates Gross Profit (GP) variation for a scenario.
    Depends on 'metrics' and 'GP_base' from main scope.
    """
    Q_new = metrics["total_volume"] * (1 + uplift)
    GP_new = Q_new * new_margin
    return (GP_new / GP_base) - 1.0

In [ ]:
# =============================================================================
# MAIN SCRIPT EXECUTION
# =============================================================================

# Step 1: Setup and Filters
create_filter_widgets(df_clean)
filters = get_filters()
df_filtered = apply_filters(df_clean, filters)

# Step 2: Margin Column Selection Widget
margin_options = ["theoretical_unit_margin", "previous_theoretical_unit_margin"]
dbutils.widgets.dropdown(
    name="margins_method",
    defaultValue="theoretical_unit_margin",
    choices=margin_options,
    label="Select Margin Column")
selected_margin_col = dbutils.widgets.get("margins_method")

# Step 3: Base Metrics Aggregation
metrics = aggregate_metrics(
    df_filtered,
    volume_col="Quantity",
    elasticity_col="final_elasticity",
    margin_col=selected_margin_col,
    zscore_threshold=2)

# Step 4: Simulation Parameters
lift_min = float(dbutils.widgets.get("lift_min"))
margin_min = float(dbutils.widgets.get("margin_min"))

# Step 5: Discount Range Calculation
discounts = calculate_discount_ranges(
    avg_elasticity=metrics["avg_elasticity"],
    avg_margin=metrics["avg_margin"],
    lift_min=lift_min,
    margin_min=margin_min)

# Step 6: Initial Results Display
display_discounts(discounts)
display(df_filtered)

# Step 7: Scenario Simulation
uplift_ideal = predict_lift(discounts["delta_star"], metrics["avg_elasticity"])
uplift_min = predict_lift(discounts["delta_min"], metrics["avg_elasticity"])
uplift_max = predict_lift(discounts["delta_max"], metrics["avg_elasticity"])

m_original = metrics["avg_margin"]
margem_ideal = (m_original + discounts["delta_star"]) / (1 + discounts["delta_star"])
margem_min = (m_original + discounts["delta_min"]) / (1 + discounts["delta_min"])
margem_max = (m_original + discounts["delta_max"]) / (1 + discounts["delta_max"])

# Profitability = (M + delta) * (1 + delta)^E
E = metrics["avg_elasticity"]
m = metrics["avg_margin"]
rentabilidade_ideal = (m + discounts["delta_star"]) * (1 + discounts["delta_star"])**E
rentabilidade_min = (m + discounts["delta_min"]) * (1 + discounts["delta_min"])**E
rentabilidade_max = (m + discounts["delta_max"]) * (1 + discounts["delta_max"])**E

# Step 8: Gross Profit (GP) Variation Calculation
GP_base = metrics["total_volume"] * metrics["avg_margin"]

var_gp_ideal = compute_gp_variation(uplift_ideal, margem_ideal)
var_gp_min = compute_gp_variation(uplift_min, margem_min)
var_gp_max = compute_gp_variation(uplift_max, margem_max)

COGS = abs(df_filtered['total_unit_cogs'].mean())
BASE_PRICE = df_filtered['List_Price'].mean()
DD = 0.095
D_STAR_PERC = -discounts['delta_star'] * 100
D_MIN_PERC = -discounts['delta_min'] * 100
D_MAX_PERC = -discounts['delta_max'] * 100

df_simulation = simulate_elasticity(BASE_PRICE, COGS, E, DD, max_discount_perc=round(D_STAR_PERC + (0.2 * D_STAR_PERC),0), steps=1000)
simulated_optimum = df_simulation.loc[df_simulation['Total_Profit_Index'].idxmax()]
calculated_opt_discount = simulated_optimum['Discount_Perc']

results, fig_simulacao = plot_simulation(
    df=df_simulation, 
    d_calc=calculated_opt_discount,
    d_star=D_STAR_PERC,
    d_min=D_MIN_PERC,
    d_max=D_MAX_PERC,
    lift_min=lift_min,        
    margin_min=margin_min,
    brands=filters['BRAND'])


# Step 9: Info DataFrame Construction
delta_gp_max_abs = results 
delta_gp_max = -delta_gp_max_abs      

uplift_gp_max = (1 + delta_gp_max)**E - 1
margin_gp_max = (m + delta_gp_max) / (1 + delta_gp_max)
rentab_gp_max = (m + delta_gp_max) * (1 + delta_gp_max)**E
var_gp_max_gp = compute_gp_variation(uplift_gp_max, margin_gp_max)

info = [
    {
        "Metric": "Theoretical Optimal Discount",
        "Value": f"{discounts['delta_star']:.2%}",
        "Explanation": (
            "Price discount calculated to balance margin reduction and sales gain.<br> "
            "Considers price sensitivity (elasticity), seeking maximum economic efficiency."),
        "Expected Uplift": f"{uplift_ideal:.2%}",
        "Estimated Margin": f"{margem_ideal:.2%}",
        "Expected Profitability": f"{rentabilidade_ideal:.2%}"
    },
    {
        "Metric": "Discount to reach Min Uplift",
        "Value": f"{discounts['delta_min']:.2%}",
        "Explanation": (
            "Minimum price variation needed to reach target volume lift (lift_min).<br>"
            "Translates volume target into discount percentage using elasticity."),
        "Expected Uplift": f"{uplift_min:.2%}",
        "Estimated Margin": f"{margem_min:.2%}",
        "Expected Profitability": f"{rentabilidade_min:.2%}"
    },
    {
        "Metric": "Safe Max Discount Limit",
        "Value": f"{discounts['delta_max']:.2%}",
        "Explanation": (
            "Highest discount allowed without compromising minimum margin (m_min).<br>"
            "Safety operational limit."),
        "Expected Uplift": f"{uplift_max:.2%}",
        "Estimated Margin": f"{margem_max:.2%}",
        "Expected Profitability": f"{rentabilidade_max:.2%}"
    },
    {
        "Metric": "Max Gross Profit Discount",
        "Value": f"{delta_gp_max:.2%}",
        "Explanation": (
            "Discount that maximizes total Gross Profit.<br>"
            "Maximizes real-world profit considering full response curve."),
        "Expected Uplift": f"{uplift_gp_max:.2%}",
        "Estimated Margin": f"{margin_gp_max:.2%}",
        "Expected Profitability": f"{rentab_gp_max:.2%}"
    }]

df_discounts_info = pd.DataFrame(info)
df_discounts_info["GP Variation"] = [
    f"{var_gp_ideal:.2%}",   
    f"{var_gp_min:.2%}",     
    f"{var_gp_max:.2%}",     
    f"{var_gp_max_gp:.2%}"]

display(df_discounts_info)

# Step 10: Display Aggregated Metrics
display(pd.DataFrame([metrics]))

# Step 11: Distribution Visualizations (KDE)
generate_histogram(df=df_filtered, numeric_col='final_elasticity', categorical_col='SALES_CHANNEL_TEAM', width=900, height=450, nbins=None, show_kde=True, show_bars=False)
generate_histogram(df=df_filtered, numeric_col='theoretical_unit_margin', categorical_col='SALES_CHANNEL_TEAM', width=900, height=450, nbins=None, show_kde=True, show_bars=False)

generate_histogram(df=df_filtered, numeric_col='final_elasticity', categorical_col='Region', width=900, height=450, nbins=None, show_kde=True, show_bars=False)
generate_histogram(df=df_filtered, numeric_col='theoretical_unit_margin', categorical_col='Region', width=900, height=450, nbins=None, show_kde=True, show_bars=False)

# Step 12: Geo Visualization
generate_geo_map(df=df_filtered, state_col='STATE', numeric_col='final_elasticity', width=800, height=550)
generate_geo_map(df=df_filtered, state_col='STATE', numeric_col='theoretical_unit_margin', width=800, height=550)

# Step 13: Historical Discount Analysis
df_filtered_spark = spark.createDataFrame(df_filtered)
df_filtered_spark.createOrReplaceTempView("df_filtered_view")

# Query to fetch applied discount joining with original table
query = f"""
SELECT
  f.*,
  t.ORDER_DISCOUNT  -- column to add
FROM df_filtered_view AS f
LEFT JOIN {table_target} AS t
  ON f.PRODUCT_ID = t.PRODUCT_ID
     AND f.BUSINESS_UNIT = t.BUSINESS_UNIT
     AND f.SUB_BUSINESS_UNIT = t.SUB_BUSINESS_UNIT
     AND f.BRAND = t.BRAND
     AND f.STATE = t.STATE
     AND f.Sub_Channel = t.Sub_Channel
     AND f.sales_team_channel_name = t.sales_team_channel_name
     AND f.SALES_CHANNEL_TEAM = t.SALES_CHANNEL_TEAM
     AND f.Product_Category = t.Product_Category
     AND f.Region = t.Region
     AND f.theoretical_unit_margin = t.theoretical_unit_margin
     AND f.month = t.month
     AND f.day_of_week = t.day_of_week
     AND f.week_of_year = t.week_of_year
"""

df_with_discount_spark = spark.sql(query)
df_discounts = df_with_discount_spark.toPandas()[['ORDER_DISCOUNT']]
df_discounts['ORDER_DISCOUNT'] = df_discounts['ORDER_DISCOUNT'].astype(float)

display(df_discounts)

# Step 14: Historical Discount Boxplot
plot_boxplot_with_stats(df_discounts, 'ORDER_DISCOUNT')

plot_simulation_plotly(df_simulation, d_calc=calculated_opt_discount, d_star=D_STAR_PERC, d_min=D_MIN_PERC, d_max=D_MAX_PERC, lift_min=lift_min, margin_min=margin_min, 
                            graph_width=1000, graph_height=600)


## 4. Manual filters and checks

In [ ]:
# 4. Applying filters
def apply_filters(df: pd.DataFrame, filters: dict) -> pd.DataFrame:
    mask = pd.Series(True, index=df.index)
    for col, sel in filters.items():
        if sel is not None:
            mask &= df[col].isin(sel)
    return df.loc[mask]

In [ ]:
import pandas as pd
from scipy import stats

def aggregate_metrics_adjusted(
    df: pd.DataFrame,
    volume_col: str = "Quantity",
    elasticity_col: str = "final_elasticity",
    margin_col: str = "theoretical_unit_margin",
    cogs_col: str = "total_unit_cogs",
    price_col: str = "List_Price",
    discount_col: str = "ORDER_DISCOUNT",
    zscore_threshold: float = 3.0) -> dict:
    """
    Adjusted aggregation including discounts, COGS and List Price.
    """
    df_clean = df.copy()

    zs_volume = stats.zscore(df_clean[volume_col])
    zs_elast = stats.zscore(df_clean[elasticity_col])
    zs_margin = stats.zscore(df_clean[margin_col])

    mask = (
        (abs(zs_volume) <= zscore_threshold) &
        (abs(zs_elast) <= zscore_threshold) &
        (abs(zs_margin) <= zscore_threshold))
    df_filtered = df_clean[mask]

    total_volume = df_filtered[volume_col].sum()

    if total_volume == 0:
        return {
            "total_volume": 0, 
            "avg_elasticity": 0.0, 
            "avg_margin": 0.0,
            "avg_cogs_abs": 0.0,
            "avg_list_price": 0.0,
            "avg_order_discount": 0.0
        }

    weighted_elasticity = float((df_filtered[elasticity_col] * df_filtered[volume_col]).sum() / total_volume)
    weighted_margin = float((df_filtered[margin_col] * df_filtered[volume_col]).sum() / total_volume)
    weighted_cogs = float((df_filtered[cogs_col] * df_filtered[volume_col]).sum() / total_volume)
    weighted_price = float((df_filtered[price_col] * df_filtered[volume_col]).sum() / total_volume)

    df_desc_active = df_filtered[df_filtered[discount_col] != 0]
    vol_desc_active = df_desc_active[volume_col].sum()

    if vol_desc_active > 0:
        weighted_discount = float((df_desc_active[discount_col] * df_desc_active[volume_col]).sum() / vol_desc_active)
    else:
        weighted_discount = 0.0

    return {
        "total_volume": int(total_volume),
        "avg_elasticity": weighted_elasticity,
        "avg_margin": weighted_margin,
        "avg_cogs_abs": abs(weighted_cogs),
        "avg_list_price": weighted_price,
        "avg_order_discount": weighted_discount
    }

In [ ]:
import numpy as np
import pandas as pd

def simulate_and_calc_elasticity(
    base_price: float, 
    cogs: float, 
    elasticity: float, 
    distributor_discount: float, 
    current_discount: float = 0.0,         
    incremental_lift_target: float = 0.05, 
    lift_min: float = 0.05, 
    margin_min: float = 0.05,
    steps: int = 100, 
    simulation_buffer: float = 0.20,
    use_exponential: bool = False) -> tuple[dict, pd.DataFrame]:
    """
    Simulates scenarios and calculates optimal ranges dynamically.
    Includes incremental analysis over status quo.

    Returns:
        dict: key metrics (delta_star, delta_min, etc.)
        DataFrame: simulation data
    """
    
    # 1. Base Variables
    net_price_base = base_price
    gp_base = net_price_base - cogs
    base_margin_standard = gp_base / base_price if base_price > 0 else 0
    constant_price = base_price * (1 - distributor_discount)
    total_profit_base = gp_base * 1.0

    # 2. Analytical Calculations
    abs_E = abs(elasticity) if elasticity != 0 else 0.0001
    
    if not use_exponential:
        delta_star = (base_margin_standard - (1 / abs_E)) / 2.0
    else:
        if abs_E > 1:
            delta_star = (abs_E / (abs_E + 1.0)) * base_margin_standard
        else:
            delta_star = 0.0

    if not use_exponential:
        delta_min = lift_min / abs_E
    else:
        delta_min = 1 - (1 / ((1 + lift_min) ** (1 / abs_E)))

    if constant_price > 0 and base_price > 0:
        target_profit_currency = margin_min * constant_price
        target_price = target_profit_currency + cogs
        delta_max = 1 - (target_price / base_price)
    else:
        delta_max = 0.0
    
    delta_max = max(delta_max, 0.0)

    # 3. Incremental Target Calc
    d_current_decimal = current_discount / 100.0 if abs(current_discount) > 1.0 else current_discount

    # Current Volume Index (Status Quo)
    if use_exponential:
        vol_index_current = (1 - d_current_decimal) ** elasticity
    else:
        vol_index_current = 1.0 + (elasticity * -d_current_decimal)
        if vol_index_current < 0: vol_index_current = 0.0001

    vol_index_target = vol_index_current * (1 + incremental_lift_target)

    # Reverse engineer needed discount
    if use_exponential:
        delta_incremental = 1 - (vol_index_target ** (1 / elasticity))
    else:
        delta_incremental = (1.0 - vol_index_target) / elasticity

    discounts_dict = {
        "delta_star": delta_star, 
        "delta_min": delta_min,
        "delta_max": delta_max,
        "delta_incremental_target": delta_incremental 
    }

    # 4. Dynamic Range Def
    critical_points = [delta_star, delta_min, delta_max, delta_incremental, d_current_decimal, 0.0]
    
    min_val = min(critical_points)
    max_val = max(critical_points)
    
    start = min_val - simulation_buffer
    end = max_val + simulation_buffer
    
    discount_range = np.linspace(start, end, steps)
    
    # 5. Numerical Simulation
    results = []
    
    for d in discount_range:
        net_price_new = base_price * (1 - d)
        gp_new = net_price_new - cogs
        
        if constant_price == 0:
            mb_new = 0
        else:
            mb_new = gp_new / constant_price
            
        if use_exponential:
            vol_new_index = (1 - d) ** elasticity
        else:
            vol_new_index = 1.0 + (elasticity * -d)
            if vol_new_index < 0: vol_new_index = 0

        # Uplift relative to list price
        perc_uplift_volume = vol_new_index - 1.0
        
        # Incremental Uplift
        if vol_index_current > 0:
            perc_uplift_incremental = (vol_new_index / vol_index_current) - 1.0
        else:
            perc_uplift_incremental = 0.0

        total_profit_new = gp_new * vol_new_index
        total_profit_index = (total_profit_new / total_profit_base) * 100 if total_profit_base != 0 else 0
        
        results.append({
            "Discount_Perc": d * 100, 
            "New_Final_Price": net_price_new,
            "Volume_Uplift_Perc": perc_uplift_volume * 100,
            "Incremental_Uplift_Perc": perc_uplift_incremental * 100, 
            "Gross_Margin_Perc": mb_new * 100,
            "Total_Profit_Index": total_profit_index
        })
        
    return discounts_dict, pd.DataFrame(results)


def plot_simulation_adjusted(df, d_calc=None, d_star=None, d_min=None, d_max=None, 
                              d_incremental=None, 
                              lift_min=None, lift_incremental=None,
                              margin_min=None, brands='GENERIC',
                              elasticity=None, margin=None, avg_discount=None):
    """
    Plots simulation results with background zones.
    """

    with plt.ioff():
        fig, ax1 = plt.subplots(figsize=(16, 7))

        # --- Zoning Limits ---
        x_min, x_max = df['Discount_Perc'].min(), df['Discount_Perc'].max()

        # --- Visual Zoning ---
        ax1.axvspan(x_min, 0, color='#C5DCE7', alpha=0.3) 
        ax1.axvspan(0, x_max, color='#e6ffe6', alpha=0.3)

        # --- Zero Divider ---
        ax1.axvline(x=0, color='black', linewidth=.5, alpha=0.4, linestyle='-.')
        ax1.axhline(y=0, color='black', linewidth=.5, alpha=0.4, linestyle='-.')

        # --- Main Axes ---
        ax1.set_xlabel('← Price Increase   |   0%   |   Discount Applied →', fontweight='bold', fontsize=11)
        ax1.set_ylabel('Uplift & Gross Margin (%)', color='tab:blue')
        
        ln1 = ax1.plot(df['Discount_Perc'], df['Volume_Uplift_Perc'], color='blue', linestyle='--', label='Volume Uplift (%)')
        ln2 = ax1.plot(df['Discount_Perc'], df['Gross_Margin_Perc'], color='orange', linestyle='--', label='Gross Margin (%)')
        
        ax1.tick_params(axis='y', labelcolor='tab:blue')
        
        ax1.grid(visible=True, axis='x', linestyle=':', alpha=0.5)
        ax1.grid(visible=False, axis='y') 

        # --- Secondary Axis ---
        ax2 = ax1.twinx()
        ax2.set_ylabel('Total Profit (Index Base 100)', color='tab:green', weight='bold')
        
        ln3 = ax2.plot(df['Discount_Perc'], df['Total_Profit_Index'], color='green', linewidth=3, label='Total Profit (Index)')
        
        ax2.tick_params(axis='y', labelcolor='tab:green')
        ax2.grid(False)

        # --- Max Point ---
        ln_dot = []
        if d_calc is not None:
            idx_max = df['Total_Profit_Index'].idxmax()
            ponto_max = df.loc[idx_max]
            ln_dot = ax2.plot(ponto_max['Discount_Perc'], ponto_max['Total_Profit_Index'], 'go', markersize=10, label='Max GP Point')

        # --- Lines and Annotations ---
        y_lim_ax1 = ax1.get_ylim()[1]
        
        def annotate_line(x_val, color, label, y_pos_factor):
            if x_val is not None:
                ax1.axvline(x=x_val, color=color, linestyle='--', linewidth=1, alpha=0.8)
                ax1.text(x_val, y_lim_ax1 * y_pos_factor, f'{x_val:.1f}%\n({label})', 
                         color=color, 
                         fontsize=9, 
                         ha='right' if x_val > 0 else 'left', va='top',
                         bbox=dict(facecolor='white', alpha=0.6, edgecolor='none', pad=1))

        annotate_line(d_star, 'red', 'Max GP', 0.9)
        annotate_line(d_max, 'brown', 'Min Margin', 0.65)
        annotate_line(d_incremental, 'darkcyan', 'Incr. Lift', 0.50)

        # --- Legend ---
        plt.title(f'Pricing Strategy Simulation - {brands}')
        plt.subplots_adjust(right=0.75)
        
        legend_handles = ln1 + ln2 + ln3 + ln_dot
        legend_labels = [h.get_label() for h in legend_handles]
        
        fig.legend(legend_handles, legend_labels, 
                   loc='lower right', 
                   bbox_to_anchor=(0.95, 0.15), 
                   fontsize=10, frameon=True, shadow=True, borderpad=1)

        # --- Segment Param Annotation ---
        if (elasticity is not None and margin is not None and avg_discount is not None 
            and lift_min is not None and lift_incremental is not None and margin_min is not None):
            
            text_info = (f"Segment Parameters:\n"
                          f"Elasticity: {elasticity}\n"
                          f"Avg Hist. Margin: {margin}\n"
                          f"Avg Hist. Disc: {avg_discount}\n"
                          f"\nCalc Parameters:\n"
                          f"Target Incr. Lift: {round(lift_incremental * 100, 1)}%\n" 
                          f"Min Margin: {round(margin_min * 100, 1)}%")
                        
            fig.text(0.81, 0.35, text_info, 
                     fontsize=10, ha='left', va='bottom',
                     bbox=dict(boxstyle="round", facecolor='white', alpha=0.9, edgecolor='lightgray'))
            
    return fig

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

def analyze_sensitivity_elasticity(
    base_price: float,
    cogs: float,
    current_elasticity: float = None, 
    brands: str = "General",
    elasticity_range: tuple = (-0.5, -6.0),
    steps: int = 200,
    use_exponential: bool = False):
    """
    Simulates Optimal Discount (Max GP) and plots reference lines.
    """
    
    e_start, e_end = elasticity_range
    elasticities = np.linspace(min(e_start, e_end), max(e_start, e_end), steps)
    
    results = []
    
    gp_base = base_price - cogs
    margin_base_full = gp_base / base_price if base_price > 0 else 0.0001 

    for e in elasticities:
        abs_E = abs(e) if e != 0 else 0.0001
        
        if not use_exponential:
            delta_star_raw = (margin_base_full - (1 / abs_E)) / 2.0
        else:
            if abs_E > 1:
                p_opt = cogs * (abs_E / (abs_E - 1))
                delta_star_raw = 1 - (p_opt / base_price)
            else:
                delta_star_raw = -0.50 

        price_var_perc = -delta_star_raw * 100

        results.append({
            "Elasticity": e,
            "Optimal_Price_Var": price_var_perc
        })

    df = pd.DataFrame(results)

    if margin_base_full > 0:
        e_intercept = -(1 / margin_base_full)
    else:
        e_intercept = None

    with plt.ioff():
        fig, ax = plt.subplots(figsize=(16, 7))
        
        ax.plot(df['Elasticity'], df['Optimal_Price_Var'], color='#2c3e50', linewidth=2.5, label='Price Variation for Max GP')
        
        ax.fill_between(df['Elasticity'], 0, df['Optimal_Price_Var'], 
                        where=(df['Optimal_Price_Var'] >= 0), 
                        color='#e74c3c', alpha=0.15, interpolate=True, label='Suggestion: Price Increase')
        
        ax.fill_between(df['Elasticity'], 0, df['Optimal_Price_Var'], 
                        where=(df['Optimal_Price_Var'] < 0), 
                        color='#27ae60', alpha=0.15, interpolate=True, label='Suggestion: Discount (Decrease)')

        ax.axhline(0, color='black', linestyle='--', linewidth=1)

        # --- VERTICAL LINES ---
        ax.axvline(-1, color='#666666', linestyle='-.', linewidth=1, label='Unit Elasticity (-1)')

        if e_intercept and min(e_start, e_end) <= e_intercept <= max(e_start, e_end):
            ax.axvline(e_intercept, color='#d35400', linestyle='--', linewidth=1, 
                       label=f'Inversion Point (E={e_intercept:.2f})')

        if current_elasticity is not None:
            ax.axvline(current_elasticity, color='#3DD6D0', linestyle='-', linewidth=1, 
                       label=f'Current Elasticity (E={current_elasticity:.2f})')

        ax.yaxis.set_major_formatter(mtick.PercentFormatter())
        ax.set_xlabel('Price Elasticity', fontsize=12, fontweight='bold')
        ax.set_ylabel('Suggested Price Variation (%)', fontsize=12, fontweight='bold')
        
        ax.set_title(f'Elasticity vs. Price Variation for Max GP\n{brands}', 
                     fontsize=14, pad=15)
        
        ax.grid(True, linestyle=':', alpha=0.6)

        ax.legend(loc='upper left', bbox_to_anchor=(1.01, 1), borderaxespad=0, frameon=True, shadow=True, fontsize=11)
        
        plt.tight_layout()

    return fig, df

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

def analyze_sensitivity_cogs(
    base_price: float,
    elasticity: float,
    current_cogs: float,
    brands: str = "General",
    cogs_range: tuple = (10.0, 80.0), 
    steps: int = 200,
    use_exponential: bool = False):
    """
    Simulates Optimal Price Variation varying COGS.
    """
    
    c_start, c_end = cogs_range
    cogs_vals = np.linspace(min(c_start, c_end), max(c_start, c_end), steps)
    
    results = []
    abs_E = abs(elasticity) if elasticity != 0 else 0.0001

    for cogs_sim in cogs_vals:
        
        gp_sim = base_price - cogs_sim
        margin_base_sim = gp_sim / base_price if base_price > 0 else 0.0001

        if not use_exponential:
            delta_star_raw = (margin_base_sim - (1 / abs_E)) / 2.0
        else:
            if abs_E > 1:
                p_opt = cogs_sim * (abs_E / (abs_E - 1))
                delta_star_raw = 1 - (p_opt / base_price)
            else:
                delta_star_raw = -0.50 

        price_var_perc = -delta_star_raw * 100

        results.append({
            "COGS_Simulated": cogs_sim,
            "Margin_Simulated": margin_base_sim,
            "Optimal_Price_Var": price_var_perc
        })

    df = pd.DataFrame(results)

    if not use_exponential:
        cogs_intercept = base_price * (1 - (1/abs_E))
    else:
        if abs_E > 1:
            cogs_intercept = base_price * ((abs_E - 1) / abs_E)
        else:
            cogs_intercept = None

    with plt.ioff():
        fig, ax = plt.subplots(figsize=(16, 7))
        
        ax.plot(df['COGS_Simulated'], df['Optimal_Price_Var'], color='#8e44ad', linewidth=2.5, label='Price Variation for Max GP')
        
        ax.fill_between(df['COGS_Simulated'], 0, df['Optimal_Price_Var'], 
                        where=(df['Optimal_Price_Var'] >= 0), 
                        color='#e74c3c', alpha=0.15, interpolate=True, label='Suggestion: Price Increase')
        
        ax.fill_between(df['COGS_Simulated'], 0, df['Optimal_Price_Var'], 
                        where=(df['Optimal_Price_Var'] < 0), 
                        color='#27ae60', alpha=0.15, interpolate=True, label='Suggestion: Discount (Decrease)')

        ax.axhline(0, color='black', linestyle='--', linewidth=1)

        ax.axvline(current_cogs, color='blue', linestyle='-', linewidth=1, 
                   label=f'Current COGS ($ {current_cogs:.2f})')

        if cogs_intercept and min(c_start, c_end) <= cogs_intercept <= max(c_start, c_end):
            ax.axvline(cogs_intercept, color='#d35400', linestyle='--', linewidth=1, 
                       label=f'Strategy Inversion (COGS ~ $ {cogs_intercept:.2f})')

        ax.yaxis.set_major_formatter(mtick.PercentFormatter())
        
        ax.set_xlabel('COGS', fontsize=12, fontweight='bold')
        ax.set_ylabel('Suggested Price Variation (%)', fontsize=12, fontweight='bold')
        
        ax.set_title(f'COGS vs. Max GP Variation\n{brands}', 
                     fontsize=14, pad=15)
        
        ax.grid(True, linestyle=':', alpha=0.6)

        ax.legend(loc='upper left', bbox_to_anchor=(1.01, 1), borderaxespad=0, frameon=True, shadow=True, fontsize=11)
        
        plt.tight_layout()

    return fig, df

In [ ]:
# Mockup test
a = 'Product_Alpha'
a.upper()

In [ ]:
####################################################################
# FILTERING
####################################################################
brand = ['Product_A']

filters = {
  'STATE': None,
 'Region': None,
 'SALES_CHANNEL_TEAM': None,
 'Sub_Channel': None,
 'sales_team_channel_name': None,
 'Product_Category': None,
 'PRODUCT_ID': None,
 'BRAND': brand,
 'BUSINESS_UNIT': None,
 'SUB_BUSINESS_UNIT': None,
 'month': None
 }

df_filtered = apply_filters(df_clean, filters)

metrics = aggregate_metrics_adjusted(
    df_filtered,
    volume_col="Quantity",
    elasticity_col="final_elasticity",
    margin_col='theoretical_unit_margin',
    cogs_col='total_unit_cogs',
    discount_col='ORDER_DISCOUNT',
    price_col='List_Price',
    zscore_threshold=3)

margin_min = metrics['avg_margin'] * 0.90 
lift_min = 0.13 

margin = metrics['avg_margin']
E = metrics['avg_elasticity']
COGS = metrics['avg_cogs_abs']
BASE_PRICE = metrics['avg_list_price']
DD = 0.095 
incremental_lift_target = 0.05 

####################################################################
# MAIN SIMULATION
####################################################################
metrics_dict, df_simulation = simulate_and_calc_elasticity(
    base_price=BASE_PRICE,
    cogs=COGS,
    elasticity=E,
    distributor_discount=DD,
    current_discount=metrics['avg_order_discount'],
    incremental_lift_target=incremental_lift_target,
    lift_min=lift_min,
    margin_min=margin_min,
    simulation_buffer=0.2,
    steps=1000,             
    use_exponential=False)

fig_sim = plot_simulation_adjusted(
    df=df_simulation, 
    d_calc=metrics_dict['delta_star'] * 100,
    d_star=metrics_dict['delta_star'] * 100,
    d_min=metrics_dict['delta_min'] * 100,
    d_max=metrics_dict['delta_max'] * 100,
    d_incremental=metrics_dict['delta_incremental_target'] * 100,
    lift_incremental=incremental_lift_target,
    lift_min=lift_min,        
    margin_min=margin_min,     
    brands=filters['BRAND'],
    elasticity=round(E, 2), 
    margin=f"{round(margin * 100, 1)}%", 
    avg_discount=f"{round(metrics['avg_order_discount'], 1)}%")

####################################################################
# SCENARIOS E/COGS SIMULATION
####################################################################
E_min = E * 1.5
E_max = E * 0.5

fig_sim_elasticities, data_sim_elasticities = analyze_sensitivity_elasticity(
    base_price=BASE_PRICE,
    cogs=COGS,
    current_elasticity=E,
    brands=filters['BRAND'],
    elasticity_range = (E_min, E_max),
    steps=1000,
    use_exponential=False)

cogs_min = COGS - (COGS * 0.7)
cogs_max = COGS + (COGS * 0.7)

fig_cogs, data_cogs = analyze_sensitivity_cogs(
    base_price=BASE_PRICE,
    elasticity=round(E, 2),
    current_cogs=COGS,
    brands=filters['BRAND'],
    cogs_range=(cogs_min, cogs_max), 
    use_exponential=False)

print('Max GP Discount')
display(df_simulation.loc[[(df_simulation['Discount_Perc'] - (metrics_dict['delta_star'] * 100)).abs().idxmin()]].round(1))

print('\n\n Incremental Uplift Target Discount')
display(df_simulation.loc[[(df_simulation['Discount_Perc'] - (metrics_dict['delta_incremental_target'] * 100)).abs().idxmin()]].round(1))

print('\n\n Min Margin Safety Discount')
display(df_simulation.loc[[(df_simulation['Discount_Perc'] - (metrics_dict['delta_max'] * 100)).abs().idxmin()]].round(1))

display(fig_sim)
display(fig_sim_elasticities)
display(fig_cogs)

In [ ]:
display(df_simulation)

In [ ]:
metrics_dict

In [ ]:
metrics

In [ ]:
plt.close(fig_sim)
plt.close(fig_sim_elasticities)
plt.close(fig_cogs)

In [ ]:
df_filtered['ORDER_DISCOUNT'].astype(float).describe()

In [ ]:
plt.close(fig_sim)
plt.close(fig_sim_elasticities)
plt.close(fig_cogs)

In [ ]:
# Anonymized data for plot
raw_data = [
    ['Product_A',       11.3,   -14.8,   -15.5,   -9.8],
    ['Product_B',         33.3,   -16.7,   -15.5,   -9.8],
    ['Product_C',     60.7,   -30.6,   -25.1,   -20.1],
    ['Product_D',       47.2,   -23.8,   -21.9,   -14.1],
    ['Product_E',        39.3,   -19.5,   -17.9,   -11.2],
    ['Product_F',        -4.0,   -14.7,   -18.6,   -9.7],
    ['Product_G',      -12.4,   -14.0,   -18.6,   -10.6],
    ['Product_H',      -9.8,   -78.4,   -72.7,   -71.3],
    ['Product_I',  -7.4,   -76.6,   -71.9,   -68.8],
    ['Product_J',     -3.3,   -98.4,   -89.8,   -89.6],
    ['Product_K',       -1.8,   -86.5,   -79.2,   -78.2],
    ['Product_L',     -4.8,   -96.7,   -87.7,   -88.0],]

data = {
    'Brand':                   [row[0] for row in raw_data],
    'Ideal_Disc':              [row[1] for row in raw_data],
    'Incr_Uplift_Disc':        [row[2] for row in raw_data],
    'Margin_Disc':             [row[3] for row in raw_data],
    'Avg_Discount':            [row[4] for row in raw_data]
}
df = pd.DataFrame(data)

cols_calc = ['Ideal_Disc', 'Incr_Uplift_Disc', 'Margin_Disc']
df['Min_Calc'] = df[cols_calc].min(axis=1)
df['Max_Calc'] = df[cols_calc].max(axis=1)

sns.set_theme(style="whitegrid")
plt.figure(figsize=(12, 7))

color_ideal = '#77dd77'   
color_uplift = '#779ecb'  
color_margin = '#ffb347'  
color_hist = '#a9a9a9'    
color_bar = '#e0e0e0'     

plt.hlines(y=df.index, xmin=df['Min_Calc'], xmax=df['Max_Calc'],
           color=color_bar, linewidth=10, label='Calc Amplitude')

plt.scatter(df['Ideal_Disc'], df.index, color=color_ideal, marker='o', s=120, label='Max GP Optimization', zorder=3, edgecolors='white', alpha=0.85)
plt.scatter(df['Incr_Uplift_Disc'], df.index, color=color_uplift, marker='v', s=120, label='Target Incr. Uplift (5%)', zorder=3, edgecolors='white', alpha=0.85)
plt.scatter(df['Margin_Disc'], df.index, color=color_margin, marker='s', s=120, label='Min Margin Target (-10%)', zorder=3, edgecolors='white', alpha=0.85)
plt.scatter(df['Avg_Discount'], df.index, color=color_hist, marker='D', s=100, label='Historical', zorder=4, edgecolors='white', alpha=0.85)

plt.axvline(0, color='#cccccc', linestyle='--', linewidth=1, zorder=1)

plt.yticks(df.index, df['Brand'], fontsize=11)
plt.xlabel('Price Variation (%)', fontsize=12)
plt.title('Calculated Discount Ranges', fontsize=14, pad=20)

plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', borderaxespad=0., frameon=False)

sns.despine(left=True, bottom=True)
plt.grid(axis='x', linestyle=':', alpha=0.6)
plt.gca().invert_yaxis()

plt.tight_layout()
plt.savefig('discount_ranges_plot.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import pandas as pd
import plotly.graph_objects as go

cols_calc = ['Ideal_Disc', 'Incr_Uplift_Disc', 'Margin_Disc']
df['Min_Calc'] = df[cols_calc].min(axis=1)
df['Max_Calc'] = df[cols_calc].max(axis=1)

# --- 2. VISUAL CONFIG ---
color_ideal = '#77dd77'   
color_uplift = '#779ecb'  
color_margin = '#ffb347'  
color_hist = '#a9a9a9'    
color_bar = '#e0e0e0'     

fig = go.Figure()

x_lines = []
y_lines = []
for index, row in df.iterrows():
    x_lines.extend([row['Min_Calc'], row['Max_Calc'], None]) 
    y_lines.extend([row['Brand'], row['Brand'], None])

fig.add_trace(go.Scatter(
    x=x_lines,
    y=y_lines,
    mode='lines',
    line=dict(color=color_bar, width=12),
    name='Calc Amplitude',
    hoverinfo='skip'))

fig.add_trace(go.Scatter(
    x=df['Ideal_Disc'],
    y=df['Brand'],
    mode='markers',
    name='Max GP Optimization',
    marker=dict(color=color_ideal, symbol='circle', size=12, line=dict(color='white', width=1), opacity=0.9),
    hovertemplate='<b>%{y}</b><br>GP: %{x}%<extra></extra>'))

fig.add_trace(go.Scatter(
    x=df['Incr_Uplift_Disc'],
    y=df['Brand'],
    mode='markers',
    name='Target Incr. Uplift (5%)',
    marker=dict(color=color_uplift, symbol='triangle-down', size=12, line=dict(color='white', width=1), opacity=0.9),
    hovertemplate='<b>%{y}</b><br>Uplift: %{x}%<extra></extra>'))

fig.add_trace(go.Scatter(
    x=df['Margin_Disc'],
    y=df['Brand'],
    mode='markers',
    name='Min Margin Target (-10%)',
    marker=dict(color=color_margin, symbol='square', size=11, line=dict(color='white', width=1), opacity=0.9),
    hovertemplate='<b>%{y}</b><br>Margin: %{x}%<extra></extra>'))

fig.add_trace(go.Scatter(
    x=df['Avg_Discount'],
    y=df['Brand'],
    mode='markers',
    name='Historical',
    marker=dict(color=color_hist, symbol='diamond', size=10, line=dict(color='white', width=1), opacity=0.9),
    hovertemplate='<b>%{y}</b><br>Historical: %{x}%<extra></extra>'))

# --- 3. FINAL LAYOUT ---
fig.update_layout(
    title='Calculated Discount Ranges',
    xaxis_title='Price Variation (%)',
    template='plotly_white',
    height=700,
    width=1100, 
    
    xaxis=dict(
        showgrid=True, 
        gridcolor='lightgray',
        zeroline=True, 
        zerolinecolor='#cccccc', 
        zerolinewidth=2),
    yaxis=dict(
        showgrid=False,
        autorange="reversed"),
    
    legend=dict(
        orientation="v",   
        yanchor="top",     
        y=1,               
        xanchor="left",    
        x=1.02,            
        bgcolor="rgba(255,255,255,0.5)"),
    
    margin=dict(r=220))

fig.show()

In [ ]:
fig.write_html("interactive_discount_chart.html")

In [ ]:
# Looking for brands with positive delta_star
def calculate_delta_star_all_brands(df_input, filters_struct):
    results = []
    
    unique_brands = df_input['BRAND'].unique()
    
    for brand in unique_brands:
        current_filters = filters_struct.copy()
        current_filters['BRAND'] = [brand]
        
        try:
            df_filtered = apply_filters(df_input, current_filters)
        except Exception:
            continue 

        metrics = aggregate_metrics_adjusted(
            df_filtered,
            volume_col="Quantity",
            elasticity_col="final_elasticity",
            margin_col='theoretical_unit_margin',
            cogs_col='total_unit_cogs',
            discount_col='ORDER_DISCOUNT',
            price_col='List_Price',
            zscore_threshold=3)

        if metrics['total_volume'] == 0 or metrics['avg_list_price'] == 0:
            results.append({
                'BRAND': brand, 
                'delta_star': None, 
                'info': 'Insufficient Data'
            })
            continue

        margin_min = metrics['avg_margin'] * 0.95
        lift_min = 0.13
        dd = 0.095
        
        try:
            metrics_dict, _ = simulate_and_calc_elasticity(
                base_price=metrics['avg_list_price'],
                cogs=metrics['avg_cogs_abs'],
                elasticity=metrics['avg_elasticity'],
                distributor_discount=dd,
                lift_min=lift_min,
                margin_min=margin_min,
                simulation_buffer=0.2,
                steps=1000,
                use_exponential=False)
            
            results.append({
                'BRAND': brand,
                'delta_star': metrics_dict.get('delta_star'),
                'avg_elasticity': metrics['avg_elasticity'],
                'avg_margin': metrics['avg_margin'],
                'avg_price': metrics['avg_list_price'],
                'total_volume': metrics['total_volume']
            })
            
        except Exception as e:
            results.append({'BRAND': brand, 'delta_star': None, 'info': f'Sim Error: {e}'})

    return pd.DataFrame(results)

base_filters = {
    'STATE': None, 'Region': None, 'SALES_CHANNEL_TEAM': None, 'Sub_Channel': None,
    'sales_team_channel_name': None, 'Product_Category': None, 'PRODUCT_ID': None,
    'BRAND': None, 'BUSINESS_UNIT': None, 'SUB_BUSINESS_UNIT': None, 'month': None
}

df_delta_star = calculate_delta_star_all_brands(df_clean, base_filters)

display(df_delta_star.sort_values(by='delta_star', ascending=False, na_position='last'))

In [ ]:
display(df_delta_star[df_delta_star.delta_star > 0].sort_values(by='total_volume', ascending=False, na_position='last'))

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

def plot_top_bottom_metrics(df_results):
    """
    Returns figure with Top/Bottom 5 Delta Star and Elasticity.
    """
    with plt.ioff():
        sns.set_style("whitegrid")
        fig, axes = plt.subplots(2, 1, figsize=(14, 14))
        
        metrics_config = [
            {
                'col': 'delta_star', 
                'title': 'Top 5 Highest and Lowest Max GP Discounts',
                'xlabel': 'Delta Star',
                'ax': axes[0],
                'color_high': 'forestgreen',
                'color_low': 'firebrick'
            },
            {
                'col': 'avg_elasticity', 
                'title': 'Top 5 Highest and Lowest Elasticities',
                'xlabel': 'Avg Elasticity',
                'ax': axes[1],
                'color_high': 'steelblue',
                'color_low': 'purple'
            }]

        for config in metrics_config:
            col = config['col']
            ax = config['ax']
            
            df_sorted = df_results.dropna(subset=[col]).sort_values(by=col, ascending=False)
            
            if df_sorted.empty:
                ax.text(0.5, 0.5, 'No data', ha='center')
                continue

            top_5 = df_sorted.head(5).copy()
            top_5['Group'] = 'Top 5 (Highest)'
            
            bottom_5 = df_sorted.tail(5).copy()
            bottom_5['Group'] = 'Bottom 5 (Lowest)'
            
            plot_df = pd.concat([top_5, bottom_5])
            
            sns.barplot(
                data=plot_df, 
                x=col, 
                y='BRAND', 
                hue='Group', 
                dodge=False, 
                ax=ax,
                palette={
                    'Top 5 (Highest)': config['color_high'], 
                    'Bottom 5 (Lowest)': config['color_low']
                })
            
            ax.set_title(config['title'], fontsize=14, fontweight='bold')
            ax.set_xlabel(config['xlabel'], fontsize=12)
            ax.set_ylabel('Brand', fontsize=12)
            
            ax.legend(loc='upper left', bbox_to_anchor=(1, 1))
            
            for container in ax.containers:
                ax.bar_label(container, fmt='%.3f', padding=3)

        plt.tight_layout()
        
        return fig

fig_brands = plot_top_bottom_metrics(df_delta_star)
fig_brands


## 5. Elasticity Analysis

In [ ]:
E

In [ ]:
df_filtered[['local_elasticity', 'final_elasticity']].describe()

In [ ]:
df_elasticity_describe = df_clean.groupby("BRAND")["final_elasticity"].describe().reset_index()
df_elasticity_describe

In [ ]:
df_elasticity_describe.sort_values("mean", ascending=True)

In [ ]:
df_clean.final_elasticity.describe()

In [ ]:
df_clean.info()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.close('all') 

features_cat = ['BRAND', 'STATE', 'BUSINESS_UNIT', 'SUB_BUSINESS_UNIT', 'Product_Category', 
    'Sub_Channel', 'sales_team_channel_name', 
    'SALES_CHANNEL_TEAM', 'month']

target = 'final_elasticity'

global_mean = df_clean[target].mean()

print(f"--- Global Elasticity Mean: {global_mean:.4f} ---")
print("(Note: More negative values indicate higher price sensitivity)\n")

for col in features_cat:
    group_stats = df_clean.groupby(col)[target].agg(['mean', 'count']).reset_index()
    
    group_stats['delta'] = group_stats['mean'] - global_mean
    
    group_stats = group_stats[group_stats['count'] > 50].sort_values('delta', ascending=False)
    
    if group_stats.empty:
        print(f"Variable {col} has insufficient data (n > 50).")
        continue

    top_positive = group_stats.head(10)
    top_negative = group_stats.tail(10)
    
    plot_data = pd.concat([top_positive, top_negative]).drop_duplicates()
    
    fig, ax = plt.subplots(figsize=(12, 7))
    
    sns.barplot(
        x='delta', 
        y=col, 
        data=plot_data, 
        ax=ax, 
        palette='vlag')
    
    ax.set_title(f'Impact on Elasticity: {col}\n(Compared to Global Mean {global_mean:.2f})')
    ax.axvline(0, color='black', linestyle='--', linewidth=1)
    
    ax.set_xlabel('Delta (Left = More Sensitive | Right = Less Sensitive)')
    
    ax.set_ylabel(col)
    ax.grid(axis='x', linestyle=':', alpha=0.5)
    
    plt.tight_layout()
    plt.show()
    
    print(f"Highlights for {col} (Sorted Less Sensitive to More Sensitive):")
    print(plot_data[[col, 'mean', 'delta', 'count']].to_string(index=False))
    print("-" * 60 + "\n")

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import OrdinalEncoder

plt.close('all')

df_model = df_clean[features_cat + [target]].copy()
encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
df_model[features_cat] = encoder.fit_transform(df_model[features_cat].astype(str))

X = df_model[features_cat]
y = df_model[target]

rf = RandomForestRegressor(n_estimators=50, max_depth=10, random_state=42, n_jobs=-1)
rf.fit(X, y)

feature_importance = pd.DataFrame({
    'feature': features_cat,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)

fig2, ax2 = plt.subplots(figsize=(10, 5))

sns.barplot(x='importance', y='feature', data=feature_importance, ax=ax2) 

ax2.set_title('Which features influence Elasticity the most?')

plt.show()

In [ ]:
import pandas as pd

peaks = {
    'Elastic Peak (-1.2)': (-1.75, -1.0),
    'Central Peak (Median)': (-1.01, -0.49),
    'Inelastic Peak (-0.5)': (-0.5, 0)
}

vars_analysis = features_cat

print("=== PEAK COMPOSITION ANALYSIS ===")

top_drivers = {} 

for peak_name, (min_val, max_val) in peaks.items():
    df_peak = df_clean[(df_clean['final_elasticity'] >= min_val) & 
                       (df_clean['final_elasticity'] <= max_val)]
    
    if df_peak.empty:
        continue
        
    print(f"\n>>> {peak_name} (Range: {min_val} to {max_val})")
    print(f"Total records in peak: {len(df_peak)}")
    
    best_var = ''
    highest_concentration = 0
    dominant_name = ''
    
    for col in vars_analysis:
        top = df_peak[col].value_counts(normalize=True).head(1)
        if not top.empty:
            val = top.values[0]
            cat = top.index[0]
            print(f"   - {col}: {cat} represents {val:.1%}")
            
            if val > highest_concentration:
                highest_concentration = val
                best_var = col
                dominant_name = cat

    top_drivers[peak_name] = (best_var, dominant_name, highest_concentration)

In [ ]:
top_drivers

In [ ]:
df_clean['final_elasticity'].describe()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import gaussian_kde
from matplotlib.lines import Line2D 

plt.close('all')

var_color = None

fig, ax = plt.subplots(figsize=(14, 8))

sns.histplot(
    data=df_clean, 
    x='final_elasticity', 
    bins=60, 
    kde=False, 
    stat="density", 
    hue=var_color, 
    multiple="stack", 
    palette='viridis',
    edgecolor='white',
    linewidth=0.5,
    ax=ax)

seaborn_legend = ax.get_legend()
dict_handles_labels = {}

if seaborn_legend:
    for h, l in zip(seaborn_legend.legendHandles, [t.get_text() for t in seaborn_legend.get_texts()]):
        dict_handles_labels[l] = h
    seaborn_legend.remove()

mean_val = df_clean['final_elasticity'].mean()
median_val = df_clean['final_elasticity'].median()

ax.axvline(mean_val, color='red', linestyle='--', linewidth=2)
ax.axvline(median_val, color='green', linestyle='-', linewidth=2)

line_mean_proxy = Line2D([0], [0], color='red', linestyle='--', linewidth=2)
line_median_proxy = Line2D([0], [0], color='green', linestyle='-', linewidth=2)

final_handles = list(dict_handles_labels.values()) + [line_mean_proxy, line_median_proxy]
final_labels = list(dict_handles_labels.keys()) + [f'Mean: {mean_val:.3f}', f'Median: {median_val:.3f}']

if var_color:
    ax.legend(handles=final_handles, labels=final_labels, title=f"{var_color} & Stats", 
            loc='upper right', frameon=True, shadow=True, fancybox=True)
else:
    ax.legend(handles=final_handles, labels=final_labels, title="Stats", 
            loc='upper right', frameon=True, shadow=True, fancybox=True) 

try:
    kde_func = gaussian_kde(df_clean['final_elasticity'].dropna())
    y_median = kde_func(median_val)[0]
    y_peak_12 = kde_func(-1.2)[0]
    y_peak_05 = kde_func(-0.5)[0]
except:
    y_median, y_peak_12, y_peak_05 = 0.5, 0.5, 0.5

d12 = top_drivers.get("Elastic Peak (-1.2)", ["Unknown", "Unknown", 0])
txt_12 = f"Dominated by {d12[0]}: {d12[1]}\n({d12[2]:.1%})"

d05 = top_drivers.get("Inelastic Peak (-0.5)", ["Unknown", "Unknown", 0])
txt_05 = f"Dominated by {d05[0]}: {d05[1]}\n({d05[2]:.1%})"

dmed = top_drivers.get("Central Peak (Median)", ["Unknown", "Unknown", 0])
txt_med = f"Dominated by {dmed[0]}: {dmed[1]}\n({dmed[2]:.1%})"

ax.set_title(f'Elasticity Density Distribution')
ax.set_xlabel('Final Elasticity')
ax.set_ylabel('Density')
ax.grid(True, linestyle=':', alpha=0.4)

plt.tight_layout()
plt.show()

In [ ]:
cols_combo = ['BRAND', 'STATE', 'BUSINESS_UNIT', 'SUB_BUSINESS_UNIT', 'Product_Category', 
    'Sub_Channel', 'sales_team_channel_name', 
    'SALES_CHANNEL_TEAM', 'month']

df_clean['elasticity_abs'] = df_clean['final_elasticity'].abs()

df_combo = df_clean.groupby(cols_combo).agg({
    'final_elasticity': 'mean',
    'elasticity_abs': 'mean',
    'Quantity': 'count' 
}).reset_index()

min_occurrences = 30
df_combo_filtered = df_combo[df_combo['Quantity'] >= min_occurrences].copy()

ranking_inelastic = df_combo_filtered.sort_values('elasticity_abs', ascending=True)

print(f"--- Top 5 LEAST Sensitive Combinations (Customers accept price) ---")
print(f"Considering only groups with n >= {min_occurrences} records\n")

display_cols = cols_combo + ['final_elasticity', 'Quantity']
print(ranking_inelastic[display_cols].head(5).to_string(index=False))

print("\n" + "="*80 + "\n")

print(f"--- Top 5 MOST Sensitive Combinations (Customers flee price increases) ---")
ranking_elastic = df_combo_filtered.sort_values('elasticity_abs', ascending=False)
print(ranking_elastic[display_cols].head(5).to_string(index=False))

In [ ]:
df_simulation

In [ ]:
# Variables 
variables_dict = {
    'E': E,
    'BASE_PRICE': BASE_PRICE,
    'DD': DD,    
    'COGS': COGS,
    'D_STAR_PERC': D_STAR_PERC,
    'D_MIN_PERC': D_MIN_PERC,
    'D_MAX_PERC': D_MAX_PERC,
    'lift_min': lift_min,
    'margin_min': margin_min
}

for k, v in variables_dict.items():
    print(f"{k}: {v}")

In [ ]:
df_simulation